In [1]:
%pip install pandas torch transformers peft evaluate rouge_score bert-score

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig
from peft import PeftModel

/common/home/projectgrps/CS425/CS425G9/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

def get_device():
    if torch.cuda.is_available():
        print('using cuda')
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")  # Apple Silicon
    print('using cpu')
    return torch.device("cpu")


2.9.0+cu128
12.8
True


In [32]:
# MODEL_NAME = "google/flan-t5-base"
# ADAPTER_DIR = "./flan_t5_med_lora_v7"

# def get_device():
#     if torch.cuda.is_available(): return torch.device("cuda")
#     if torch.backends.mps.is_available(): return torch.device("mps")
#     return torch.device("cpu")

# def load_model(fine_tuned: bool = True):
#     """
#     If merged_dir is provided and exists, load merged full model.
#     Else load base + LoRA adapters from adapter_dir.
#     """
#     device = get_device()
#     if fine_tuned:
#         tok = AutoTokenizer.from_pretrained(MODEL_NAME)
#         base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
#         mdl = PeftModel.from_pretrained(base, ADAPTER_DIR)

#     else:
#         tok = AutoTokenizer.from_pretrained(MODEL_NAME)
#         mdl = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)


#     mdl.to(device).eval()
#     return tok, mdl, device

# def format_prompt(question: str, patient_info: str) -> str:
#     return (
#         f"Answer this medical question: Medical Question: {question} Patient Information: {patient_info}"
#     )

# @torch.inference_mode()
# def generate_answers(tokenizer, model, device, items: list[tuple[str, str]],
#                      max_new_tokens: int = 256, min_new_tokens: int = 32,
#                      num_beams: int = 1, do_sample: bool = False,
#                      temperature: float = 0.7, top_p: float = 0.95,
#                      batch_size: int = 8):
#     """
#     items: list of (question, patient_info)
#     """
#     gen_cfg = GenerationConfig(
#         max_new_tokens=max_new_tokens,
#         min_new_tokens=min_new_tokens,
#         num_beams=num_beams,
#         do_sample=do_sample,
#         temperature=temperature if do_sample else None,
#         top_p=top_p if do_sample else None,
#         eos_token_id=tokenizer.eos_token_id,
#         pad_token_id=tokenizer.pad_token_id,
#         no_repeat_ngram_size=3,
#     )

#     outputs = []
#     for i in range(0, len(items), batch_size):
#         batch_qp = items[i:i+batch_size]
#         prompts = [format_prompt(q, p) for q, p in batch_qp]
#         enc = tokenizer(prompts, return_tensors="pt", truncation=True, padding=True).to(device)
#         out_ids = model.generate(**enc, generation_config=gen_cfg)
#         texts = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
#         # Strip the "Answer:" prefix if the model repeats it
#         cleaned = [t.replace("Answer:", "").strip() for t in texts]
#         outputs.extend(cleaned)
#     return outputs

# if __name__ == "__main__":
#     # Choose ONE:
#     # 1) Base + LoRA adapters:
#     tokenizer, model, device = load_model(fine_tuned=False)
#     # 2) Or merged full model:
#     # tokenizer, model, device = load_model(adapter_dir=None, merged_dir=MERGED_DIR)

#     cases = [
#         ("can i get pregnant if my sonography shows bulky ovaries with small follicles?",
#          "recently i had a sonography test where the report says that my both ovaries appear bulky. left ovary measures 12cc and right ovary measures 7cc and multiple small follicles are seen in both ovaries. my question to you is whether i can be pregnant or not . my age is 26yrs old"

#         )
#     ]

#     answers = generate_answers(
#         tokenizer, model, device, cases,
#         max_new_tokens=256,   # raise if your answers are long
#         min_new_tokens=48,
#         num_beams=5,          # set to 1 for speed; >1 for quality
#         do_sample=False,      # True for more varied outputs
#         batch_size=4,
#     )

#     print("===Base Model===")
#     for (q, p), a in zip(cases, answers):
#         print("\nQ:", q)
#         print("P:", p)
#         print("A:", a)
#         print("\n")

#     tokenizer, model, device = load_model(fine_tuned=True)

#     answers = generate_answers(
#         tokenizer, model, device, cases,
#         max_new_tokens=256,   # raise if your answers are long
#         min_new_tokens=48,
#         num_beams=5,          # set to 1 for speed; >1 for quality
#         do_sample=False,      # True for more varied outputs
#         batch_size=4,
#     )

#     print("===Fine Tuned Model===")
#     for (q, p), a in zip(cases, answers):
#         print("\nQ:", q)
#         print("P:", p)
#         print("A:", a)
#         print("\n")

In [4]:
medical_test = pd.read_csv("medical_qa_test.csv")

In [5]:
# We keep the question and patient info constant, and only change the model capability layer by layer: base → LoRA → LoRA + RAG.

# 1) ORIGINAL INFERENCE CODE 

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig
from peft import PeftModel
import torch
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MODEL_NAME = "google/flan-t5-base"
ADAPTER_DIR = "./flan_t5_med_lora_v7"

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

def load_model(fine_tuned: bool = True):
    """
    If merged_dir is provided and exists, load merged full model.
    Else load base + LoRA adapters from adapter_dir.
    """
    device = get_device()
    if fine_tuned:
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
        mdl = PeftModel.from_pretrained(base, ADAPTER_DIR)

    else:
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)


    mdl.to(device).eval()
    return tok, mdl, device

def format_prompt(question: str, patient_info: str) -> str:
    return (
        f"Answer this medical question: Medical Question: {question} Patient Information: {patient_info}"
    )

@torch.inference_mode()
def generate_answers(tokenizer, model, device, items: list[tuple[str, str]],
                     max_new_tokens: int = 256, min_new_tokens: int = 32,
                     num_beams: int = 1, do_sample: bool = False,
                     temperature: float = 0.7, top_p: float = 0.95,
                     batch_size: int = 8):
    """
    items: list of (question, patient_info)
    """
    gen_cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        num_beams=num_beams,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        no_repeat_ngram_size=3,
    )

    outputs = []
    for i in range(0, len(items), batch_size):
        batch_qp = items[i:i+batch_size]
        prompts = [format_prompt(q, p) for q, p in batch_qp]
        enc = tokenizer(prompts, return_tensors="pt", truncation=True, padding=True).to(device)
        out_ids = model.generate(**enc, generation_config=gen_cfg)
        texts = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        cleaned = [t.replace("Answer:", "").strip() for t in texts]
        outputs.extend(cleaned)
    return outputs

# 2) RAG CODE 

from dotenv import load_dotenv
import os, re, requests, html
from typing import List, Dict, Optional
from bs4 import BeautifulSoup

# Load API keys
load_dotenv(dotenv_path="cred.env")
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")
CSE_ID  = os.getenv("GOOGLE_CSE_ID")

TRUSTED_SITES: List[str] = ["cda.gov.sg", "medlineplus.gov", "cdc.gov", "moh.gov.sg"]
EXCLUDE_PDF   = True
HTTP_TIMEOUT  = 15
TOP_PAGES     = 1
TOP_SENTENCES = 3

STOP = set("""
a an the of to in is are was were be been being and or for on with by as at from this that it its into about
if can may might should could would will how what when where which who whom whose i you your me my our their them
they he she we us
""".split())
TOKEN = re.compile(r"\b[a-z0-9]+\b")
def tokens(s: str) -> List[str]:
    return [t for t in TOKEN.findall(s.lower()) if t not in STOP and len(t) >= 3]

def build_query(q: str, sites: Optional[List[str]]) -> str:
    if sites:
        return q + " " + " OR ".join(f"site:{s}" for s in sites)
    return q

def cse_search(full_query: str, sites: Optional[List[str]], num=10) -> List[Dict]:
    if not API_KEY or not CSE_ID:
        return []
    url = "https://www.googleapis.com/customsearch/v1"
    params = {"key": API_KEY, "cx": CSE_ID, "q": build_query(full_query, sites),
              "num": min(num, 10), "safe": "active"}
    try:
        r = requests.get(url, params=params, timeout=HTTP_TIMEOUT, verify=False)
        r.raise_for_status()
        items = r.json().get("items") or []
    except Exception:
        items = []
    results = []
    for it in items:
        link = (it.get("link") or "").strip()
        if EXCLUDE_PDF and link.lower().endswith(".pdf"):
            continue
        results.append({
            "title":   (it.get("title") or "").strip(),
            "url":     link,
            "snippet": (it.get("snippet") or "").replace("…"," ").replace("...", " ").strip(),
        })
    return results

def fetch_main_text(url: str) -> str:
    try:
        r = requests.get(url, timeout=HTTP_TIMEOUT,
                         headers={"User-Agent": "Mozilla/5.0"}, verify=False)
        if r.status_code != 200:
            return ""
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script","style","noscript","header","footer","nav","aside"]):
            tag.decompose()
        node = soup.find("main") or soup.find("article") or soup.body
        if not node:
            return ""
        text = " ".join(el.get_text(" ", strip=True)
                        for el in node.find_all(["p","li"]))
        text = re.sub(r"\s+", " ", text)
        return html.unescape(text).strip()
    except Exception:
        return ""

def split_sents(text: str) -> List[str]:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text)
            if len(s.strip()) >= 30]

def select_best_sentences(page_text: str, full_query: str, k: int) -> str:
    if not page_text:
        return ""
    sents = split_sents(page_text)
    if not sents:
        return ""
    qset = set(tokens(full_query))
    ranked = sorted(sents, key=lambda s: len(qset & set(tokens(s))), reverse=True)
    return " ".join(ranked[:k])

def build_rag_prompt(user_question: str,
                     keep_pages: int = TOP_PAGES,
                     keep_sentences: int = TOP_SENTENCES) -> Dict:
    items = cse_search(user_question, TRUSTED_SITES, num=10) or \
            cse_search(user_question, None, num=10)
    if not items:
        return {"prompt": f"question: {user_question}\nretrieve:", "sources": []}

    pages = items[:keep_pages]
    parts, sources = [], []
    for it in pages:
        page_text = fetch_main_text(it["url"])
        expanded   = select_best_sentences(page_text, user_question, keep_sentences)
        snippet    = expanded or it["snippet"]
        snippet    = " ".join(re.split(r"(?<=[.!?])\s+", snippet)[:keep_sentences])
        parts.append(f"[{it['title']}]({it['url']}): {snippet}")
        sources.append({"title": it["title"], "url": it["url"]})
    retrieve_blob = " [SEP] ".join(parts)
    prompt = f"question: {user_question}\nretrieve: {retrieve_blob}"

    #     # add debug prints
    # print("\n================= RAG DEBUG =================")
    # print("Question:", user_question)
    # print("Retrieved blob:\n", retrieve_blob)
    # print("Sources:")
    # for s in sources:
    #     print(" -", s["title"], "-", s["url"])
    # print("=============================================\n")
    
    return {"prompt": prompt, "sources": sources}

def format_prompt_rag(question: str, patient_info: str, rag_blob: str) -> str:
    return (
        f"Answer this medical question: "
        f"Medical Question: {question} "
        f"Patient Information: {patient_info} "
        f"Retrieved Information: {rag_blob}"
    )

@torch.inference_mode()
def generate_answers_rag(tokenizer, model, device, items: list[tuple[str, str]],
                         keep_pages: int = TOP_PAGES, keep_sentences: int = TOP_SENTENCES,
                         max_new_tokens: int = 256, min_new_tokens: int = 32,
                         num_beams: int = 1, do_sample: bool = False,
                         temperature: float = 0.7, top_p: float = 0.95,
                         batch_size: int = 4):
    gen_cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        num_beams=num_beams,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        no_repeat_ngram_size=3,
    )
    rag_packs = [build_rag_prompt(q, keep_pages, keep_sentences) for q, _ in items]
    prompts = [format_prompt_rag(q, p, pack["prompt"]) for (q, p), pack in zip(items, rag_packs)]
    
    for (q, p), pack in zip(items, rag_packs):
        print("\n================= RAG RETRIEVAL =================")
        print("Question:", q)
        print("Retrieved Blob:\n", pack["prompt"])
        print("Sources:")
        for s in pack["sources"]:
            print(" -", s["title"], "-", s["url"])
        print("=================================================")

    outputs = []
    for i in range(0, len(prompts), batch_size):
        enc = tokenizer(prompts[i:i+batch_size],
                        return_tensors="pt", truncation=True, padding=True).to(device)
        out_ids = model.generate(**enc, generation_config=gen_cfg)
        texts = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        cleaned = [t.replace("Answer:", "").strip() for t in texts]
        for j, ans in enumerate(cleaned):
            outputs.append({"answer": ans, "sources": rag_packs[i+j]["sources"]})
    return outputs

# 3) MAIN EXECUTION BLOCK 
if __name__ == "__main__":
    cases = [
        ("can i get pregnant if my sonography shows bulky ovaries with small follicles?",
         "recently i had a sonography test where the report says that my both ovaries appear bulky. left ovary measures 12cc and right ovary measures 7cc and multiple small follicles are seen in both ovaries. my question to you is whether i can be pregnant or not . my age is 26yrs old")
    ]
    tokenizer, model, device = load_model(fine_tuned=False)
    answers = generate_answers(
        tokenizer, model, device, cases,
        max_new_tokens=256,   # raise if answers are long
        min_new_tokens=48,
        num_beams=5,          # set to 1 for speed or >1 for quality
        do_sample=False,      # True for more varied outputs
        batch_size=4,
    )

    print("================= Base Model =================")
    for (q, p), a in zip(cases, answers):
        print("\nQuestion:", q)
        print("Patient Context:", p)
        print("Answer:", a)

    tokenizer, model, device = load_model(fine_tuned=True)

    answers = generate_answers(
        tokenizer, model, device, cases,
        max_new_tokens=256,   # raise if answers are long
        min_new_tokens=48,
        num_beams=4,          # set to 1 for speed or >1 for quality
        do_sample=False,      # True for more varied outputs
        batch_size=4,
    )

    print("================= Fine Tuned Model =================")
    for (q, p), a in zip(cases, answers):
        print("\nQuestion:", q)
        print("Patient Context:", p)
        print("Answer:", a)
    
    tokenizer, model, device = load_model(fine_tuned=True)
    print("================= Fine Tuned Model + Retrieval =================")

    rag_results = generate_answers_rag(
        tokenizer, model, device, cases,
        keep_pages=1, keep_sentences=3,
        max_new_tokens=256, min_new_tokens=48,
        num_beams=3, do_sample=False, batch_size=2
    )
    print("================= Final Answer (RAG) =================")
    for (q, p), out in zip(cases, rag_results):
        print("\nQuestion:", q)
        print("Patient Context:", p)
        print("Answer:", out["answer"])
        if out["sources"]:
            print("Sources:")
            for k, s in enumerate(out["sources"], 1):
                print(f" [{k}] {s['title']} - {s['url']}")


================= Base Model =================

Question: can i get pregnant if my sonography shows bulky ovaries with small follicles?
Patient Context: recently i had a sonography test where the report says that my both ovaries appear bulky. left ovary measures 12cc and right ovary measures 7cc and multiple small follicles are seen in both ovaries. my question to you is whether i can be pregnant or not . my age is 26yrs old
Answer: my sonography test says that my both ovaries appear bulky. left ovary measures 12cc and right ovaary measures 7cc , and multiple small follicles are seen in both ovaries. my question to you is whether i can be pregnant or not . my age is 26yrs old
================= Fine Tuned Model =================

Question: can i get pregnant if my sonography shows bulky ovaries with small follicles?
Patient Context: recently i had a sonography test where the report says that my both ovaries appear bulky. left ovary measures 12cc and right ovary measures 7cc and multiple

In [7]:
# T5-base

# ===============================================================
# === 1) BASE FLAN-T5 MODEL (NO LoRA, NO RAG) ===================
# ===============================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig
import torch
import pandas as pd
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MODEL_NAME = "google/flan-t5-base"

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

def load_model_base_only():
    """Load PURE base FLAN-T5 (no LoRA)."""
    device = get_device()
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    mdl.to(device).eval()
    return tok, mdl, device

def format_prompt(question: str, patient_info: str) -> str:
    return (
        f"Answer this medical question: "
        f"Medical Question: {question} "
        f"Patient Information: {patient_info}"
    )

@torch.inference_mode()
def generate_answers(tokenizer, model, device, items, 
                     max_new_tokens=256, min_new_tokens=32,
                     num_beams=1, do_sample=False,
                     temperature=0.7, top_p=0.95,
                     batch_size=8):

    gen_cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        num_beams=num_beams,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        no_repeat_ngram_size=3,
    )

    outputs = []
    for i in range(0, len(items), batch_size):
        batch = items[i:i+batch_size]
        prompts = [format_prompt(q, p) for q, p in batch]
        enc = tokenizer(prompts, return_tensors="pt", truncation=True, padding=True).to(device)

        out_ids = model.generate(**enc, generation_config=gen_cfg)
        decoded = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        cleaned = [x.replace("Answer:", "").strip() for x in decoded]

        outputs.extend(cleaned)
    return outputs


# ===============================================================
# === 2) CSV HELPERS ============================================
# ===============================================================

def parse_formatted_input(text: str):
    """Extract question + patient info."""
    if not isinstance(text, str): 
        return "", ""
    import re
    m = re.search(r"Medical Question:\s*(.*?)\s*Patient Information:\s*(.*)",
                  text, flags=re.I | re.S)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return text.strip(), ""

def load_cases_from_csv(csv_path: str, n: int = 21):
    df = pd.read_csv(csv_path, encoding="utf-8", on_bad_lines="skip")
    formatted = df["Formatted_Input"].fillna("")
    cases = [parse_formatted_input(x) for x in formatted.head(n)]
    return cases, df.head(n)


# ===============================================================
# === 3) MAIN: RUN BASE MODEL ON 21 ROWS =========================
# ===============================================================

if __name__ == "__main__":

    tokenizer, model, device = load_model_base_only()

    cases, raw_df = load_cases_from_csv("medical_qa_test.csv", n=21)
    print(f"\n[CSV] Loaded {len(cases)} samples.\n")

    # Run base T5 on 21 items
    model_outputs = generate_answers(
        tokenizer, model, device, cases,
        max_new_tokens=512, min_new_tokens=48,
        num_beams=3, do_sample=False, batch_size=2
    )

    # ===========================================================
    # === PRINT 20 ROWS (SKIP ROW 14 ONLY) =====================
    # ===========================================================

    print("\n================= MODEL OUTPUTS (20 rows) =================")
    for idx, ((q, p), ans) in enumerate(zip(cases, model_outputs), start=1):

        if idx == 14:   # Skip row 14
            continue

        print(f"\n=== SAMPLE {idx} ===")
        print("Question:", q)
        print("Patient Info:", p)
        print("Model Answer:", ans)

    print("\n===========================================================\n")

    # Save predictions
    rows = []
    for idx, ((q, p), answer) in enumerate(zip(cases, model_outputs), start=1):
        gold = raw_df.iloc[idx-1]["Answer"] if "Answer" in raw_df.columns else ""
        rows.append({
            "idx": idx,
            "question": q,
            "patient_info": p,
            "model_answer": answer,
            "gold_answer": gold,
        })

    import os
    os.makedirs("outputs", exist_ok=True)
    out_path = "outputs/medical_qa_pred_BASE_T5_only.csv"
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print("\nSaved predictions to:", out_path)


    # ===========================================================
    # === 4) ROUGE + BERTSCORE ==================================
    # ===========================================================

    import evaluate
    df = pd.DataFrame(rows)

    # Skip empty gold answers
    df = df[df["gold_answer"].fillna("").str.strip() != ""].reset_index(drop=True)
    print(f"Evaluating {len(df)} samples...\n")

    preds = df["model_answer"].tolist()
    refs  = df["gold_answer"].tolist()

    # ROUGE
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=preds, references=refs)

    print("\n================= ROUGE SCORES =================")
    for k, v in rouge_results.items():
        print(f"{k}: {v:.4f}")
    print("================================================\n")

    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=preds, references=refs, lang="en")
    bert_f1_mean = sum(bert_results["f1"]) / len(bert_results["f1"])

    print("================= BERTSCORE =================")
    print(f"Mean F1: {bert_f1_mean:.4f}")
    print("=============================================\n")

    # Save evaluated results
    df["rouge1"] = rouge_results.get("rouge1", 0)
    df["rouge2"] = rouge_results.get("rouge2", 0)
    df["rougeL"] = rouge_results.get("rougeL", 0)
    df["bertscore_f1"] = [round(f, 4) for f in bert_results["f1"]]

    out_eval_path = "outputs/medical_qa_pred_BASE_T5_only_evaluated.csv"
    df.to_csv(out_eval_path, index=False)
    print(f"Saved evaluated results to: {out_eval_path}")



[CSV] Loaded 21 samples.


================= MODEL OUTPUTS (20 rows) =================

=== SAMPLE 1 ===
Question: Medical Question: will multiple blows to the cheekbone result in the cheekbones growing larger?
Patient Info: i d like to direct my question to a bone specialist. this is probably a really silly question however i wanted to ask anyway; i ve been in a few fights over the last year or so and recently it got me thinking; would taking multiple blows to the cheekbone area result in my cheekbones growing larger in response to stress? i know it is a bit of an absurd question but i just wanted to ask anyway.
Model Answer: Is stress a factor in the growth of the cheekbones and how can it affect the size of your cheekbone? If so, what is the best course of action to take in order to achieve this outcome? Is it a good idea to take multiple blows to the cheek bone area after a fight to see if this is the case?

=== SAMPLE 2 ===
Question: Medical Question: q. can headache occur as a s

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


================= BERTSCORE =================
Mean F1: 0.8251

Saved evaluated results to: outputs/medical_qa_pred_BASE_T5_only_evaluated.csv


In [8]:
# T5-base + LoRA (v9)

# ===============================================================
# === 1) FLAN-T5 + LoRA ADAPTER (NO RAG) =========================
# ===============================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig
from peft import PeftModel
import torch
import pandas as pd
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MODEL_NAME = "google/flan-t5-base"
ADAPTER_DIR = "./flan_t5_med_lora_v9"   

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

def load_model_with_lora():
    """
    Load FLAN-T5 base + LoRA adapter.
    **This is NOT merged** so only LoRA weights are trainable.
    """
    device = get_device()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model = PeftModel.from_pretrained(base, ADAPTER_DIR)

    return tokenizer, model.to(device).eval(), device

def format_prompt(question: str, patient_info: str) -> str:
    return (
        f"Answer this medical question: Medical Question: {question} "
        f"Patient Information: {patient_info}"
    )

@torch.inference_mode()
def generate_answers(tokenizer, model, device, items,
                     max_new_tokens=256, min_new_tokens=32,
                     num_beams=1, do_sample=False,
                     temperature=0.7, top_p=0.95,
                     batch_size=8):

    gen_cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        num_beams=num_beams,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        no_repeat_ngram_size=3,
    )

    outputs = []
    for i in range(0, len(items), batch_size):
        batch = items[i:i+batch_size]
        prompts = [format_prompt(q, p) for q, p in batch]

        enc = tokenizer(
            prompts,
            return_tensors="pt",
            truncation=True,
            padding=True
        ).to(device)

        out_ids = model.generate(**enc, generation_config=gen_cfg)
        decoded = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        cleaned = [x.replace("Answer:", "").strip() for x in decoded]

        outputs.extend(cleaned)

    return outputs


# ===============================================================
# === 2) CSV HELPERS ============================================
# ===============================================================

def parse_formatted_input(text: str):
    """Extract question + patient info from Formatted_Input."""
    if not isinstance(text, str): return "", ""
    import re
    m = re.search(
        r"Medical Question:\s*(.*?)\s*Patient Information:\s*(.*)",
        text, flags=re.I | re.S
    )
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return text.strip(), ""

def load_cases_from_csv(csv_path: str, n: int = 21):
    df = pd.read_csv(csv_path, on_bad_lines="skip")
    formatted = df["Formatted_Input"].fillna("")
    cases = [parse_formatted_input(x) for x in formatted.head(n)]
    return cases, df.head(n)


# ===============================================================
# === 3) MAIN — RUN LORA MODEL ON 21 ROWS =======================
# ===============================================================

if __name__ == "__main__":

    tokenizer, model, device = load_model_with_lora()

    cases, raw_df = load_cases_from_csv("medical_qa_test.csv", n=21)
    print(f"\n[CSV] Loaded {len(cases)} samples.\n")

    # Run LoRA T5 on 21 rows
    model_outputs = generate_answers(
        tokenizer, model, device, cases,
        max_new_tokens=512, min_new_tokens=48,
        num_beams=3, do_sample=False, batch_size=2
    )

    # ===========================================================
    # === PRINT 20 ROWS (SKIP ROW 14 ONLY) =====================
    # ===========================================================

    print("\n================= MODEL OUTPUTS (20 rows) =================")
    for idx, ((q, p), ans) in enumerate(zip(cases, model_outputs), start=1):

        if idx == 14:   # skip row 14
            continue

        print(f"\n=== SAMPLE {idx} ===")
        print("Question:", q)
        print("Patient Info:", p)
        print("Model Answer:", ans)

    print("\n===========================================================\n")

    # Save predictions
    rows = []
    for idx, ((q, p), answer) in enumerate(zip(cases, model_outputs), start=1):
        gold = raw_df.iloc[idx-1]["Answer"] if "Answer" in raw_df.columns else ""
        rows.append({
            "idx": idx,
            "question": q,
            "patient_info": p,
            "model_answer": answer,
            "gold_answer": gold,
        })

    import os
    os.makedirs("outputs", exist_ok=True)
    out_path = "outputs/medical_qa_pred_LoRA_only.csv"
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print("\nSaved predictions to:", out_path)


    # ===========================================================
    # === 4) METRICS: ROUGE + BERTSCORE =========================
    # ===========================================================

    import evaluate
    df = pd.DataFrame(rows)

    df = df[df["gold_answer"].fillna("").str.strip() != ""].reset_index(drop=True)
    print(f"Evaluating {len(df)} samples...\n")

    preds = df["model_answer"].tolist()
    refs  = df["gold_answer"].tolist()

    # ROUGE
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=preds, references=refs)

    print("\n================= ROUGE SCORES =================")
    for k, v in rouge_results.items():
        print(f"{k}: {v:.4f}")
    print("================================================\n")

    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=preds, references=refs, lang="en")
    bert_f1_mean = sum(bert_results["f1"]) / len(bert_results["f1"])

    print("================= BERTSCORE =================")
    print(f"Mean F1: {bert_f1_mean:.4f}")
    print("=============================================\n")

    # Save evaluated results
    df["rouge1"] = rouge_results.get("rouge1", 0)
    df["rouge2"] = rouge_results.get("rouge2", 0)
    df["rougeL"] = rouge_results.get("rougeL", 0)
    df["bertscore_f1"] = [round(f, 4) for f in bert_results["f1"]]

    out_eval_path = "outputs/medical_qa_pred_LoRA_only_evaluated.csv"
    df.to_csv(out_eval_path, index=False)
    print(f"Saved evaluated results to: {out_eval_path}")



[CSV] Loaded 21 samples.


================= MODEL OUTPUTS (20 rows) =================

=== SAMPLE 1 ===
Question: Medical Question: will multiple blows to the cheekbone result in the cheekbones growing larger?
Patient Info: i d like to direct my question to a bone specialist. this is probably a really silly question however i wanted to ask anyway; i ve been in a few fights over the last year or so and recently it got me thinking; would taking multiple blows to the cheekbone area result in my cheekbones growing larger in response to stress? i know it is a bit of an absurd question but i just wanted to ask anyway.
Model Answer: thanks for your question on hcm. i can understand your concern. if the cheekbones are growing larger, then it is likely that you are having a swollen cheekbone. you should consult a bone specialist and get done an x-ray of the cheek and if there is a swelling, then you can take a topical steroid like acetaminophen. hope i have answered your query, if you have do

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


================= BERTSCORE =================
Mean F1: 0.8560

Saved evaluated results to: outputs/medical_qa_pred_LoRA_only_evaluated.csv


In [4]:
# T5-base + LoRA (v9) + RAG

# ===============================================================
# === 1) ORIGINAL INFERENCE CODE (DO NOT CHANGE THIS SECTION) ===
# ===============================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig
from peft import PeftModel
import torch
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MODEL_NAME = "google/flan-t5-base"
ADAPTER_DIR = "./flan_t5_med_lora_v9"
MERGED_DIR  = "./flan_t5_med_lora_v9"

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

def load_model(adapter_dir: str | None = ADAPTER_DIR, merged_dir: str | None = None):
    device = get_device()
    if merged_dir:
        tok = AutoTokenizer.from_pretrained(merged_dir)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(merged_dir)
    else:
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
        mdl = PeftModel.from_pretrained(base, adapter_dir)
    mdl.to(device).eval()
    return tok, mdl, device

def format_prompt(question: str, patient_info: str) -> str:
    return f"Answer this medical question: Medical Question: {question} Patient Information: {patient_info}"

@torch.inference_mode()
def generate_answers(tokenizer, model, device, items: list[tuple[str, str]],
                     max_new_tokens: int = 256, min_new_tokens: int = 32,
                     num_beams: int = 1, do_sample: bool = False,
                     temperature: float = 0.7, top_p: float = 0.95,
                     batch_size: int = 8):
    gen_cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        num_beams=num_beams,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        no_repeat_ngram_size=3,
    )

    outputs = []
    for i in range(0, len(items), batch_size):
        batch_qp = items[i:i+batch_size]
        prompts = [format_prompt(q, p) for q, p in batch_qp]
        enc = tokenizer(prompts, return_tensors="pt", truncation=True, padding=True).to(device)
        out_ids = model.generate(**enc, generation_config=gen_cfg)
        texts = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        cleaned = [t.replace("Answer:", "").strip() for t in texts]
        outputs.extend(cleaned)
    return outputs


# ===============================================================
# === 2) RAG RETRIEVAL CODE =====================================
# ===============================================================

from dotenv import load_dotenv
import os, re, requests, html
from typing import List, Dict, Optional
from bs4 import BeautifulSoup
import pandas as pd

load_dotenv(dotenv_path="cred.env")
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")
CSE_ID  = os.getenv("GOOGLE_CSE_ID")

TRUSTED_SITES: List[str] = ["cda.gov.sg", "cdc.gov", "moh.gov.sg"]
EXCLUDE_PDF   = True
HTTP_TIMEOUT  = 15
TOP_PAGES     = 1
TOP_SENTENCES = 3

STOP = set("""
a an the of to in is are was were be been being and or for on with by as at from this that it its into about
if can may might should could would will how what when where which who whom whose i you your me my our their them
they he she we us
""".split())
TOKEN = re.compile(r"\b[a-z0-9]+\b")

def tokens(s: str) -> List[str]:
    return [t for t in TOKEN.findall(s.lower()) if t not in STOP and len(t) >= 3]

def build_query(q: str, sites: Optional[List[str]]) -> str:
    if sites:
        return q + " " + " OR ".join(f"site:{s}" for s in sites)
    return q

def cse_search(full_query: str, sites: Optional[List[str]], num=10) -> List[Dict]:
    if not API_KEY or not CSE_ID:
        return []
    url = "https://www.googleapis.com/customsearch/v1"
    params = {"key": API_KEY, "cx": CSE_ID, "q": build_query(full_query, sites),
              "num": min(num, 10), "safe": "active"}
    try:
        r = requests.get(url, params=params, timeout=HTTP_TIMEOUT, verify=False)
        r.raise_for_status()
        items = r.json().get("items") or []
    except Exception:
        items = []
    results = []
    for it in items:
        link = (it.get("link") or "").strip()
        if EXCLUDE_PDF and link.lower().endswith(".pdf"):
            continue
        results.append({
            "title":   (it.get("title") or "").strip(),
            "url":     link,
            "snippet": (it.get("snippet") or "").replace("…"," ").replace("...", " ").strip(),
        })
    return results

def fetch_main_text(url: str) -> str:
    try:
        r = requests.get(url, timeout=HTTP_TIMEOUT,
                         headers={"User-Agent": "Mozilla/5.0"}, verify=False)
        if r.status_code != 200:
            return ""
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script","style","noscript","header","footer","nav","aside"]):
            tag.decompose()
        node = soup.find("main") or soup.find("article") or soup.body
        if not node:
            return ""
        text = " ".join(el.get_text(" ", strip=True)
                        for el in node.find_all(["p","li"]))
        text = re.sub(r"\s+", " ", text)
        return html.unescape(text).strip()
    except Exception:
        return ""

def split_sents(text: str) -> List[str]:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text)
            if len(s.strip()) >= 30]

def select_best_sentences(page_text: str, full_query: str, k: int) -> str:
    if not page_text:
        return ""
    sents = split_sents(page_text)
    if not sents:
        return ""
    qset = set(tokens(full_query))
    ranked = sorted(sents, key=lambda s: len(qset & set(tokens(s))), reverse=True)
    return " ".join(ranked[:k])

def build_rag_prompt(user_question: str,
                     keep_pages: int = TOP_PAGES,
                     keep_sentences: int = TOP_SENTENCES) -> Dict:
    items = cse_search(user_question, TRUSTED_SITES, num=10) or \
            cse_search(user_question, None, num=10)
    if not items:
        return {"prompt": f"question: {user_question}\nretrieve:", "sources": []}

    pages = items[:keep_pages]
    parts, sources = [], []
    for it in pages:
        page_text = fetch_main_text(it["url"])
        expanded   = select_best_sentences(page_text, user_question, keep_sentences)
        snippet    = expanded or it["snippet"]
        snippet    = " ".join(re.split(r"(?<=[.!?])\s+", snippet)[:keep_sentences])
        parts.append(f"[{it['title']}]({it['url']}): {snippet}")
        sources.append({"title": it["title"], "url": it["url"]})
    retrieve_blob = " [SEP] ".join(parts)
    prompt = f"question: {user_question}\nretrieve: {retrieve_blob}"
    return {"prompt": prompt, "sources": sources}

def format_prompt_rag(question: str, patient_info: str, rag_blob: str) -> str:
    return f"Answer this medical question: Medical Question: {question} Patient Information: {patient_info} Retrieved Information: {rag_blob}"

@torch.inference_mode()
def generate_answers_rag(tokenizer, model, device, items: list[tuple[str, str]],
                         keep_pages: int = TOP_PAGES, keep_sentences: int = TOP_SENTENCES,
                         max_new_tokens: int = 256, min_new_tokens: int = 32,
                         num_beams: int = 1, do_sample: bool = False,
                         temperature: float = 0.7, top_p: float = 0.95,
                         batch_size: int = 4):
    gen_cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        num_beams=num_beams,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        no_repeat_ngram_size=3,
    )
    rag_packs = [build_rag_prompt(q, keep_pages, keep_sentences) for q, _ in items]
    prompts = [format_prompt_rag(q, p, pack["prompt"]) for (q, p), pack in zip(items, rag_packs)]

    outputs = []
    for i in range(0, len(prompts), batch_size):
        enc = tokenizer(prompts[i:i+batch_size],
                        return_tensors="pt", truncation=True, padding=True).to(device)
        out_ids = model.generate(**enc, generation_config=gen_cfg)
        texts = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        cleaned = [t.replace("Answer:", "").strip() for t in texts]
        for j, ans in enumerate(cleaned):
            outputs.append({"answer": ans, "sources": rag_packs[i+j]["sources"]})
    return outputs


# ===================== CSV HELPERS ==============================
def parse_formatted_input(s: str):
    if not isinstance(s, str): return "", ""
    m = re.search(r"Medical Question:\s*(.*?)\s*Patient Information:\s*(.*)", s, flags=re.I | re.S)
    if m: return m.group(1).strip(), m.group(2).strip()
    parts = re.split(r"Patient Information:\s*", s, flags=re.I)
    if len(parts) == 2:
        q = re.sub(r"^Answer this medical question:\s*", "", parts[0], flags=re.I).strip()
        return q, parts[1].strip()
    return s.strip(), ""

def load_cases_from_csv(csv_path: str, n: int = 5):
    df = pd.read_csv(csv_path, encoding="utf-8", on_bad_lines="skip")
    if "Formatted_Input" not in df.columns:
        raise ValueError("CSV must contain a 'Formatted_Input' column.")
    df_small = df.head(n).copy()
    formatted = df_small["Formatted_Input"].fillna("")
    cases = [parse_formatted_input(x) for x in formatted]
    return cases, df_small


# ===============================================================
# === 3) MAIN EXECUTION BLOCK ==================================
# ===============================================================
if __name__ == "__main__":
    tokenizer, model, device = load_model(adapter_dir=ADAPTER_DIR, merged_dir=None)

    cases, raw_df = load_cases_from_csv("medical_qa_test.csv", n=21)
    print("\n[CSV] Loaded rows:", len(cases))

    rag_results = generate_answers_rag(
        tokenizer, model, device, cases,
        keep_pages=1, keep_sentences=3,
        max_new_tokens=512, min_new_tokens=48,
        num_beams=3, do_sample=False, batch_size=2
    )

    print("\n================= MODEL OUTPUTS (20 rows) =================")
    for idx, ((q, p), out) in enumerate(zip(cases, rag_results), start=1):
        # Skip row 14 → index 13 (no gold answer)
        if idx == 14: 
            continue
    
        print(f"\n=== SAMPLE {idx} ===")
        print("Question:", q)
        print("Patient Info:", p)
        print("Model Answer:", out['answer'])
        
        srcs = " | ".join(
            f"{s['title']}<{s['url']}>" for s in out["sources"]
        ) if out["sources"] else ""
    
        if srcs:
            print("Sources:", srcs)
    
    print("\n===========================================================\n")

    rows = []
    for idx, ((q, p), out) in enumerate(zip(cases, rag_results), start=1):
        gold = raw_df.iloc[idx-1]["Answer"] if "Answer" in raw_df.columns else ""
        srcs = " | ".join(f"{s['title']}<{s['url']}>" for s in out["sources"]) if out["sources"] else ""
        rows.append({
            "idx": idx,
            "question": q,
            "patient_info": p,
            "model_answer": out["answer"],
            "gold_answer": gold,
            "sources": srcs,
        })

    os.makedirs("outputs", exist_ok=True)
    out_path = "outputs/medical_qa_pred_sample.csv"
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print("\n Saved predictions to:", out_path)

    # ===============================================================
    # === 4) EVALUATION METRICS: ROUGE + BERTSCORE (SKIP EMPTY GOLD) ==
    # ===============================================================
    import evaluate
    import pandas as pd
    
    # Load metrics
    rouge = evaluate.load("rouge")
    bertscore = evaluate.load("bertscore")
    
    # Convert to DataFrame (if not already)
    df = pd.DataFrame(rows)
    
    # --- Filter: skip rows without gold answers ---
    df = df[df["gold_answer"].fillna("").str.strip() != ""].reset_index(drop=True)
    print(f"🧹 Skipped rows without gold answers. Evaluating {len(df)} samples.\n")
    
    # Prepare data
    preds = df["model_answer"].tolist()
    refs  = df["gold_answer"].tolist()
    
    # ---- Compute ROUGE ----
    rouge_results = rouge.compute(predictions=preds, references=refs)
    print("\n================= ROUGE SCORES =================")
    for k, v in rouge_results.items():
        print(f"{k}: {v:.4f}")
    print("================================================\n")
    
    # ---- Compute BERTScore ----
    bert_results = bertscore.compute(predictions=preds, references=refs, lang="en")
    bert_f1_mean = round(sum(bert_results["f1"]) / len(bert_results["f1"]), 4)
    
    print("================= BERTSCORE =================")
    print(f"Mean F1: {bert_f1_mean}")
    print("=============================================\n")
    
    # ---- Merge metrics back into DataFrame ----
    df["rouge1"] = rouge_results.get("rouge1", 0)
    df["rouge2"] = rouge_results.get("rouge2", 0)
    df["rougeL"] = rouge_results.get("rougeL", 0)
    df["bertscore_f1"] = [round(f, 4) for f in bert_results["f1"]]
    
    # ---- Save updated file ----
    out_path_filtered = "outputs/medical_qa_pred_sample_evaluated.csv"
    df.to_csv(out_path_filtered, index=False)
    print(f"Saved filtered + evaluated results to: {out_path_filtered}")


[CSV] Loaded rows: 21

================= MODEL OUTPUTS (20 rows) =================

=== SAMPLE 1 ===
Question: Medical Question: will multiple blows to the cheekbone result in the cheekbones growing larger?
Patient Info: i d like to direct my question to a bone specialist. this is probably a really silly question however i wanted to ask anyway; i ve been in a few fights over the last year or so and recently it got me thinking; would taking multiple blows to the cheekbone area result in my cheekbones growing larger in response to stress? i know it is a bit of an absurd question but i just wanted to ask anyway.
Model Answer: thanks for your question on hcm. i can understand your concern. if you have a severe headache or stiff neck, you may need a x ray to rule out a head injury. a complete x-ray of the head and a CT scan of the skull will help in determining the severity of the injury. hope i have answered your query. let me know if i could assist you further.
Sources: Head injury - fir

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


================= BERTSCORE =================
Mean F1: 0.8497

Saved filtered + evaluated results to: outputs/medical_qa_pred_sample_evaluated.csv


In [9]:
# T5-base + RAG

# ===============================================================
# === 1) ORIGINAL INFERENCE CODE — MODIFIED FOR BASE MODEL ONLY ==
# ===============================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig
import torch
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MODEL_NAME = "google/flan-t5-base"

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

def load_model_base_only():
    """
    Load PURE base FLAN-T5 (NO LORA)
    """
    device = get_device()
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    mdl.to(device).eval()
    return tok, mdl, device

def format_prompt(question: str, patient_info: str) -> str:
    return f"Answer this medical question: Medical Question: {question} Patient Information: {patient_info}"

@torch.inference_mode()
def generate_answers(tokenizer, model, device, items: list[tuple[str, str]],
                     max_new_tokens: int = 256, min_new_tokens: int = 32,
                     num_beams: int = 1, do_sample: bool = False,
                     temperature: float = 0.7, top_p: float = 0.95,
                     batch_size: int = 8):
    
    gen_cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        num_beams=num_beams,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        no_repeat_ngram_size=3,
    )

    outputs = []
    for i in range(0, len(items), batch_size):
        batch_qp = items[i:i+batch_size]
        prompts = [format_prompt(q, p) for q, p in batch_qp]
        enc = tokenizer(prompts, return_tensors="pt", truncation=True, padding=True).to(device)
        out_ids = model.generate(**enc, generation_config=gen_cfg)
        texts = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        cleaned = [t.replace("Answer:", "").strip() for t in texts]
        outputs.extend(cleaned)
    return outputs


# ===============================================================
# === 2) RAG RETRIEVAL CODE =====================================
# ===============================================================

from dotenv import load_dotenv
import os, re, requests, html
from typing import List, Dict, Optional
from bs4 import BeautifulSoup
import pandas as pd

load_dotenv(dotenv_path="cred.env")
load_dotenv()

API_KEY = os.getenv("GOOGLE_API_KEY")
CSE_ID  = os.getenv("GOOGLE_CSE_ID")

TRUSTED_SITES = ["medlineplus.gov", "cdc.gov", "moh.gov.sg", "cda.gov.sg"]
EXCLUDE_PDF   = True
HTTP_TIMEOUT  = 15
TOP_PAGES     = 1
TOP_SENTENCES = 3

STOP = set("""
a an the of to in is are was were be been being and or for on with by as at from this that it its into about
if can may might should could would will how what when where which who whom whose i you your me my our their them
they he she we us
""".split())

TOKEN = re.compile(r"\b[a-z0-9]+\b")

def tokens(s: str) -> List[str]:
    return [t for t in TOKEN.findall(s.lower()) if t not in STOP and len(t) >= 3]

def build_query(q: str, sites: Optional[List[str]]) -> str:
    if sites:
        return q + " " + " OR ".join(f"site:{s}" for s in sites)
    return q

def cse_search(full_query: str, sites: Optional[List[str]], num=10) -> List[Dict]:
    if not API_KEY or not CSE_ID:
        return []
    url = "https://www.googleapis.com/customsearch/v1"
    params = {
        "key": API_KEY, "cx": CSE_ID, "q": build_query(full_query, sites),
        "num": min(num, 10), "safe": "active"
    }
    try:
        r = requests.get(url, params=params, timeout=HTTP_TIMEOUT, verify=False)
        r.raise_for_status()
        items = r.json().get("items") or []
    except Exception:
        items = []

    results = []
    for it in items:
        link = (it.get("link") or "").strip()
        if EXCLUDE_PDF and link.lower().endswith(".pdf"):
            continue
        results.append({
            "title": it.get("title", "").strip(),
            "url": link,
            "snippet": it.get("snippet", "").replace("…", " ").strip(),
        })
    return results

def fetch_main_text(url: str) -> str:
    try:
        r = requests.get(
            url, timeout=HTTP_TIMEOUT,
            headers={"User-Agent": "Mozilla/5.0"}, verify=False
        )
        if r.status_code != 200:
            return ""
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script","style","noscript","header","footer","nav","aside"]):
            tag.decompose()
        node = soup.find("main") or soup.find("article") or soup.body
        if not node:
            return ""
        text = " ".join(el.get_text(" ", strip=True) for el in node.find_all(["p","li"]))
        text = re.sub(r"\s+", " ", text)
        return html.unescape(text).strip()
    except Exception:
        return ""

def split_sents(text: str) -> List[str]:
    return [
        s.strip() for s in re.split(r"(?<=[.!?])\s+", text)
        if len(s.strip()) >= 30
    ]

def select_best_sentences(page_text: str, full_query: str, k: int) -> str:
    if not page_text:
        return ""
    sents = split_sents(page_text)
    if not sents:
        return ""
    qset = set(tokens(full_query))
    ranked = sorted(sents, key=lambda s: len(qset & set(tokens(s))), reverse=True)
    return " ".join(ranked[:k])

def build_rag_prompt(user_question: str,
                     keep_pages: int = TOP_PAGES,
                     keep_sentences: int = TOP_SENTENCES) -> Dict:
    
    items = cse_search(user_question, TRUSTED_SITES, num=10) \
            or cse_search(user_question, None, num=10)

    if not items:
        return {"prompt": f"question: {user_question}\nretrieve:", "sources": []}

    pages = items[:keep_pages]
    parts, sources = [], []

    for it in pages:
        page_text = fetch_main_text(it["url"])
        expanded  = select_best_sentences(page_text, user_question, keep_sentences)
        snippet   = expanded or it["snippet"]
        snippet   = " ".join(re.split(r"(?<=[.!?])\s+", snippet)[:keep_sentences])
        parts.append(f"[{it['title']}]({it['url']}): {snippet}")
        sources.append({"title": it["title"], "url": it["url"]})

    retrieve_blob = " [SEP] ".join(parts)
    prompt = f"question: {user_question}\nretrieve: {retrieve_blob}"
    return {"prompt": prompt, "sources": sources}

def format_prompt_rag(question: str, patient_info: str, rag_blob: str) -> str:
    return (
        f"Answer this medical question: Medical Question: {question} "
        f"Patient Information: {patient_info} Retrieved Information: {rag_blob}"
    )

@torch.inference_mode()
def generate_answers_rag(tokenizer, model, device, items: list[tuple[str, str]],
                         keep_pages: int = TOP_PAGES, keep_sentences: int = TOP_SENTENCES,
                         max_new_tokens: int = 256, min_new_tokens: int = 32,
                         num_beams: int = 1, do_sample: bool = False,
                         temperature: float = 0.7, top_p: float = 0.95,
                         batch_size: int = 4):

    gen_cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        num_beams=num_beams,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        no_repeat_ngram_size=3,
    )

    rag_packs = [build_rag_prompt(q, keep_pages, keep_sentences) for q, _ in items]
    prompts = [
        format_prompt_rag(q, p, pack["prompt"])
        for (q, p), pack in zip(items, rag_packs)
    ]

    outputs = []
    for i in range(0, len(prompts), batch_size):

        enc = tokenizer(
            prompts[i:i+batch_size],
            return_tensors="pt", truncation=True, padding=True
        ).to(device)

        out_ids = model.generate(**enc, generation_config=gen_cfg)
        decoded = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        cleaned = [x.replace("Answer:", "").strip() for x in decoded]

        for j, ans in enumerate(cleaned):
            outputs.append({
                "answer": ans,
                "sources": rag_packs[i+j]["sources"]
            })

    return outputs


# ===============================================================
# === 3) CSV HELPERS ============================================
# ===============================================================

def parse_formatted_input(s: str):
    if not isinstance(s, str): return "", ""
    m = re.search(
        r"Medical Question:\s*(.*?)\s*Patient Information:\s*(.*)",
        s, flags=re.I | re.S
    )
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return s.strip(), ""

def load_cases_from_csv(csv_path: str, n: int = 50):
    df = pd.read_csv(csv_path, encoding="utf-8", on_bad_lines="skip")
    formatted_inputs = df["Formatted_Input"].fillna("")
    cases = [parse_formatted_input(text) for text in formatted_inputs.head(n)]
    return cases, df.head(n)


# ===============================================================
# === 4) MAIN — BASE FLAN-T5 ONLY + RAG ==========================
# ===============================================================

if __name__ == "__main__":

    # Load base FLAN-T5 (no LoRA)
    tokenizer, model, device = load_model_base_only()

    # Load CSV test cases
    cases, raw_df = load_cases_from_csv("medical_qa_test.csv", n=21)
    print(f"\n[CSV] Loaded {len(cases)} samples.\n")

    # Run RAG
    rag_results = generate_answers_rag(
        tokenizer, model, device, cases,
        keep_pages=1, keep_sentences=3,
        max_new_tokens=512, min_new_tokens=48,
        num_beams=3, do_sample=False, batch_size=2
    )

    print("\n================= MODEL OUTPUTS (20 rows) =================")
    for idx, ((q, p), out) in enumerate(zip(cases, rag_results), start=1):
        # Skip row 14 → index 13 (no gold answer)
        if idx == 14: 
            continue
    
        print(f"\n=== SAMPLE {idx} ===")
        print("Question:", q)
        print("Patient Info:", p)
        print("Model Answer:", out['answer'])
        
        srcs = " | ".join(
            f"{s['title']}<{s['url']}>" for s in out["sources"]
        ) if out["sources"] else ""
    
        if srcs:
            print("Sources:", srcs)
    
    print("\n===========================================================\n")

    # Save predictions
    rows = []
    for idx, ((q, p), out) in enumerate(zip(cases, rag_results), start=1):
        gold = raw_df.iloc[idx-1]["Answer"] if "Answer" in raw_df.columns else ""
        srcs = " | ".join(f"{s['title']}<{s['url']}>" for s in out["sources"])
        rows.append({
            "idx": idx,
            "question": q,
            "patient_info": p,
            "model_answer": out["answer"],
            "gold_answer": gold,
            "sources": srcs,
        })

    os.makedirs("outputs", exist_ok=True)
    out_path = "outputs/medical_qa_pred_sample_BASE+RAG.csv"
    pd.DataFrame(rows).to_csv(out_path, index=False)

    print("\nSaved predictions to:", out_path)


    # ===========================================================
    # === 5) ROUGE + BERTSCORE EVALUATION =======================
    # ===========================================================
    import evaluate

    df = pd.DataFrame(rows)

    # Skip empty-gold rows
    df = df[df["gold_answer"].fillna("").str.strip() != ""].reset_index(drop=True)
    print(f"Evaluating {len(df)} samples...\n")

    preds = df["model_answer"].tolist()
    refs  = df["gold_answer"].tolist()

    # ROUGE
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=preds, references=refs)

    print("\n================= ROUGE SCORES =================")
    for k, v in rouge_results.items():
        print(f"{k}: {v:.4f}")
    print("================================================\n")

    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=preds, references=refs, lang="en")
    bert_f1_mean = float(sum(bert_results["f1"])) / len(bert_results["f1"])

    print("================= BERTSCORE =================")
    print(f"Mean F1: {bert_f1_mean:.4f}")
    print("=============================================\n")

    # Save evaluated file
    df["rouge1"] = rouge_results.get("rouge1", 0)
    df["rouge2"] = rouge_results.get("rouge2", 0)
    df["rougeL"] = rouge_results.get("rougeL", 0)
    df["bertscore_f1"] = [round(f, 4) for f in bert_results["f1"]]

    out_eval_path = "outputs/medical_qa_pred_sample_BASE+RAG_evaluated.csv"
    df.to_csv(out_eval_path, index=False)
    print(f"Saved evaluated results to: {out_eval_path}")



[CSV] Loaded 21 samples.


================= MODEL OUTPUTS (20 rows) =================

=== SAMPLE 1 ===
Question: Medical Question: will multiple blows to the cheekbone result in the cheekbones growing larger?
Patient Info: i d like to direct my question to a bone specialist. this is probably a really silly question however i wanted to ask anyway; i ve been in a few fights over the last year or so and recently it got me thinking; would taking multiple blows to the cheekbone area result in my cheekbones growing larger in response to stress? i know it is a bit of an absurd question but i just wanted to ask anyway.
Model Answer: [Head injury - first aid: MedlinePlus Medical Encyclopedia](https://medlineplus.gov/ency/article/000028.htm): The head may look fine, but problems could result from bleeding or swelling inside the skull. Get medical help right away if the person: Becomes very sleepy Behaves abnormally, or has speech that does not make sense Develops a severe headache or stiff ne

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


================= BERTSCORE =================
Mean F1: 0.8153

Saved evaluated results to: outputs/medical_qa_pred_sample_BASE+RAG_evaluated.csv
